# CRISP-DM Fase 1: Business Understanding (Comprensión del Negocio)
## Dataset: SIPSA Precios de Insumos y Fertilizantes Agrícolas (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 1: Business Understanding) | **Fase PDCO**: PLAN  
**Estándares**: DAMA-DMBOK 2 (Metadata Management & Governance), SWEBOK Cap. 1, ISO/IEC 25010.

---

### 🎯 1. Objetivo Estratégico y Problema de Negocio
Monitorear la estructura de costos de los productores agropecuarios para proyectar la viabilidad financiera del ciclo productivo.

### 💼 2. Entidad del Dominio y Alcance
* **Entidad Analítica Principal**: `Precios de Venta de Fertilizantes, Plaguicidas y Medicamentos Veterinarios`
* **Custodio y Fuente Oficial**: DANE / IDEAM / UPRA / SIPSA Colombia.
* **Granularidad Espacial**: `Municipio comercializador / Almacenes de insumos`
* **Granularidad Temporal**: `Mensual`

### 💡 3. Preguntas de Negocio Mapeadas (`docs/1-Bateria_preguntas.md`):
* **C1: Costos Directos de Insumos**
* **C2: Paridad Insumo-Producto**
* **C3: Presión Inflacionaria de Fertilizantes**

### 📊 4. Indicadores Clave de Desempeño (KPIs):
* `Índice de precios de fertilizantes`
* `Ratio de paridad insumo/cosecha`
* `Elasticidad de costo directo`

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 1: VERIFICACIÓN DEL CONTRATO DE NEGOCIO Y CONFIGURACIÓN YAML]
# ==============================================================================
import yaml

config_path = APP_ROOT / "config" / "datasets_config.yaml"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_data = yaml.safe_load(f)
    print(f"📄 Archivo de configuración cargado: {config_path.name}")
    dataset_cfg = config_data.get("datasets", {}).get("03_sipsa_insumos", {})
    print(f"📋 Especificación del Dataset en YAML: {dataset_cfg}")
else:
    print("ℹ️ Configuración YAML por defecto activa.")

print("\n--- DEFINICIÓN DEL PROBLEMA DE NEGOCIO ---")
print("• Entidad:", "Precios de Venta de Fertilizantes, Plaguicidas y Medicamentos Veterinarios")
print("• Granularidad Espacial:", "Municipio comercializador / Almacenes de insumos")
print("• Granularidad Temporal:", "Mensual")
print("• KPIs Objetivo:", ['Índice de precios de fertilizantes', 'Ratio de paridad insumo/cosecha', 'Elasticidad de costo directo'])


In [ ]:
# ==============================================================================
# [FASE 1: INSPECCIÓN DE DISPONIBILIDAD DE FUENTES Y CONTRATO DAMA-DMBOK]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
parquet_path = APP_ROOT / "data/processed/sipsa_insumos.parquet"

print("🔍 Verificación de Almacenamiento en Data Lakehouse:")
print(f"• Archivo Parquet Silver/Gold: {parquet_path.name} -> Existe: {parquet_path.exists()}")
print(f"• Base de Datos SQLite Gold: {db_path.name} -> Existe: {db_path.exists()}")

if parquet_path.exists():
    df_sample = pd.read_parquet(parquet_path)
    print(f"\n✅ Contrato de datos preliminar verificado:")
    print(f"• Filas totales: {len(df_sample):,}")
    print(f"• Columnas: {list(df_sample.columns)}")
